# EZhire - Resume-Job Semantic Similarity Scoring System (Vast.ai)
> Contextual chunk-aware SBERT fine-tuning + TF-IDF Dual-Model NLP Pipeline | Gradio Dashboard

**Instructions:** Start a Vast.ai GPU instance, open this notebook, install dependencies, then Run all. This version uses batch size 32 and is intended for GPUs with more VRAM than Colab T4.

### Pipeline
1. Install & import
2. Load dataset (`0xnbk/resume-ats-score-v1-en`)
3. Preprocess — split `[SEP]`, preserve section line breaks, clean, fixed-range normalise scores
4. **Fine-tune SBERT** with `CosineSimilarityLoss` on contextual chunk pairs
5. Batch score full validation split
6. Evaluate — Spearman, Pearson, NDCG, Precision@K, MRR, normalized MAE, raw ATS MAE/RMSE/R2
7. Launch Gradio dashboard

In [ ]:
!pip install -q sentence-transformers datasets gradio scikit-learn plotly nltk PyMuPDF

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
import nltk
import fitz
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, ndcg_score, r2_score
from scipy.stats import spearmanr, pearsonr
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('Imports OK')

In [ ]:
ds       = load_dataset('0xnbk/resume-ats-score-v1-en')
df_train = ds['train'].to_pandas()
df_val   = ds['validation'].to_pandas()

print(f'Train : {len(df_train)} rows')
print(f'Val   : {len(df_val)} rows')
print(f'Columns: {list(df_train.columns)}')
print(f'ATS score range: {df_train["ats_score"].min():.1f} - {df_train["ats_score"].max():.1f}')
print(f'Labels:\n{df_train["original_label"].value_counts().to_string()}')
print('\nSample text (first 400 chars):')
print(df_train['text'].iloc[0][:400])

In [ ]:
# text column format: '<resume> [SEP] <job_description>'
# ATS scores use the fixed dataset range 18.3 - 90.7; normalise to 0-1 for CosineSimilarityLoss.

ATS_SCORE_MIN = 18.3
ATS_SCORE_MAX = 90.7
STOP_WORDS = set(stopwords.words('english'))

def split_sep(text):
    if not isinstance(text, str): text = str(text)
    if '[SEP]' in text:
        a, b = text.split('[SEP]', 1)
        return a.strip(), b.strip()
    mid = len(text)//2
    return text[:mid].strip(), text[mid:].strip()

def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
    tokens = word_tokenize(re.sub(r'\s+', ' ', text).strip())
    return ' '.join(t for t in tokens if t not in STOP_WORDS and len(t)>1)

def raw_text(text):
    if not isinstance(text, str): text = str(text)
    return re.sub(r'\s+', ' ', text).strip()

def context_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t\f\v]+', ' ', line).strip() for line in text.split('\n')]
    return '\n'.join(line for line in lines if line)

def normalize_score(series, s_min=ATS_SCORE_MIN, s_max=ATS_SCORE_MAX):
    values = pd.to_numeric(series, errors='coerce').astype(float)
    return ((values - s_min) / (s_max - s_min)).clip(0, 1)

def denormalize_score(values, s_min=ATS_SCORE_MIN, s_max=ATS_SCORE_MAX):
    return np.asarray(values, dtype=float) * (s_max - s_min) + s_min

def build_df(source_df):
    splits = source_df['text'].apply(split_sep)
    ats = pd.to_numeric(source_df['ats_score'], errors='coerce').fillna(ATS_SCORE_MIN)
    out = pd.DataFrame({
        'resume_raw'    : splits.apply(lambda x: context_text(x[0])),
        'jd_raw'        : splits.apply(lambda x: context_text(x[1])),
        'resume_clean'  : splits.apply(lambda x: clean_text(x[0])),
        'jd_clean'      : splits.apply(lambda x: clean_text(x[1])),
        'original_label': source_df['original_label'].values,
        'ats_score_raw' : ats.values,
        'ground_truth'  : normalize_score(ats).values
    })
    return out.dropna(subset=['resume_raw','jd_raw']).reset_index(drop=True)

df_tr = build_df(df_train)
df_vl = build_df(df_val)

print(f'Train preprocessed: {len(df_tr)} rows')
print(f'Val   preprocessed: {len(df_vl)} rows')
print(f'Fixed ATS normalisation range: {ATS_SCORE_MIN:.1f} - {ATS_SCORE_MAX:.1f}')
print(f'Ground truth range (train): {df_tr["ground_truth"].min():.3f} - {df_tr["ground_truth"].max():.3f}')
print(f'Ground truth range (val):   {df_vl["ground_truth"].min():.3f} - {df_vl["ground_truth"].max():.3f}')
df_tr.head(2)

In [ ]:
# Contextual chunk-aware fine-tuning for all-mpnet-base-v2.
# all-mpnet-base-v2 has a short context window, so long resumes/JDs are
# section-prefixed token chunks before training and validation instead of being truncated.

MODEL_NAME    = 'sentence-transformers/all-mpnet-base-v2'
OUTPUT_PATH   = './ezhire-finetuned-chunked-vastai'
EPOCHS        = 4
BATCH_SIZE    = 32 if DEVICE == 'cuda' else 8
LEARNING_RATE = 2e-5
WARMUP_STEPS  = 100
WEIGHT_DECAY  = 0.01

MAX_SEQ_LENGTH          = 384
CHUNK_TOKENS            = 320
CHUNK_OVERLAP           = 64
MAX_RESUME_CHUNKS       = 10
MAX_JD_CHUNKS           = 8
MAX_TRAIN_PAIRS_PER_ROW = 4
MAX_EVAL_PAIRS_PER_ROW  = 2
DOC_EVAL_N              = 200
TOP_K_CHUNK_PAIRS       = 5
ENCODE_BATCH_SIZE       = 32 if DEVICE == 'cuda' else 8
RANDOM_SEED             = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f'Loading base model: {MODEL_NAME}')
sbert_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
sbert_model.max_seq_length = MAX_SEQ_LENGTH

SECTION_ALIASES = {
    'SUMMARY': ['summary', 'professional summary', 'profile', 'objective', 'career objective', 'about'],
    'SKILLS': ['skills', 'technical skills', 'core skills', 'key skills', 'competencies', 'core competencies', 'technologies', 'tools'],
    'EXPERIENCE': ['experience', 'work experience', 'professional experience', 'employment history', 'work history', 'career history'],
    'PROJECTS': ['projects', 'selected projects', 'project experience', 'portfolio'],
    'EDUCATION': ['education', 'academic background', 'qualifications', 'academic qualifications'],
    'CERTIFICATIONS': ['certifications', 'certification', 'licenses', 'training'],
    'REQUIREMENTS': ['requirements', 'job requirements', 'required skills', 'minimum qualifications', 'preferred qualifications', 'must have', 'must haves'],
    'RESPONSIBILITIES': ['responsibilities', 'role responsibilities', 'duties', 'what you will do', 'job description'],
    'BENEFITS': ['benefits', 'compensation', 'perks']
}
SECTION_TERMS = sorted(
    {term for terms in SECTION_ALIASES.values() for term in terms},
    key=len,
    reverse=True
)
INLINE_SECTION_RE = re.compile(
    r'\b(' + '|'.join(re.escape(term) for term in SECTION_TERMS) + r')\s*[:\-]\s*',
    flags=re.IGNORECASE
)

def canonical_section_name(header):
    cleaned = re.sub(r'[^a-z0-9 +/#&.-]', ' ', str(header).lower())
    cleaned = re.sub(r'\s+', ' ', cleaned).strip(' :-')
    if not cleaned or len(cleaned.split()) > 6:
        return None
    for label, terms in SECTION_ALIASES.items():
        if cleaned in terms:
            return label
    return None

def detect_line_section(line):
    stripped = re.sub(r'^[#*\-\u2022\s]+', '', str(line)).strip()
    if not stripped:
        return None, ''
    match = re.match(r'^([A-Za-z][A-Za-z0-9 /&+#().-]{1,45})\s*[:\-]\s*(.*)$', stripped)
    if match:
        label = canonical_section_name(match.group(1))
        if label:
            return label, match.group(2).strip()
    label = canonical_section_name(stripped)
    if label:
        return label, ''
    return None, stripped

def split_context_sections(text):
    original = '' if text is None else str(text)
    lines = [ln.strip() for ln in re.split(r'[\r\n]+', original) if ln.strip()]
    sections, current_label, buffer, found_line_header = [], 'GENERAL', [], False

    def flush():
        body = raw_text(' '.join(buffer))
        if body:
            sections.append((current_label, body))

    for line in lines:
        label, remainder = detect_line_section(line)
        if label:
            flush()
            current_label, buffer, found_line_header = label, [], True
            if remainder:
                buffer.append(remainder)
        else:
            buffer.append(remainder)
    flush()
    if found_line_header and len(lines) > 1 and sections:
        return sections

    compact = raw_text(original)
    matches = list(INLINE_SECTION_RE.finditer(compact))
    if not matches:
        return [('GENERAL', compact)] if compact else [('GENERAL', '')]

    sections = []
    prefix = raw_text(compact[:matches[0].start()])
    if prefix:
        sections.append(('GENERAL', prefix))
    for idx, match in enumerate(matches):
        label = canonical_section_name(match.group(1)) or 'GENERAL'
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(compact)
        body = raw_text(compact[start:end])
        if body:
            sections.append((label, body))
    return sections or [('GENERAL', compact)]

def token_chunk_body(body, tokenizer, max_tokens=CHUNK_TOKENS, overlap=CHUNK_OVERLAP):
    body = raw_text(body)
    if not body:
        return []
    token_ids = tokenizer.encode(body, add_special_tokens=False, truncation=False)
    if len(token_ids) <= max_tokens:
        return [body]
    overlap = min(overlap, max_tokens // 2)
    step = max_tokens - overlap
    chunks = []
    for start in range(0, len(token_ids), step):
        piece = token_ids[start:start + max_tokens]
        chunk = raw_text(tokenizer.decode(piece, skip_special_tokens=True))
        if chunk:
            chunks.append(chunk)
        if start + max_tokens >= len(token_ids):
            break
    return chunks

def make_text_chunks(text, tokenizer, max_tokens=CHUNK_TOKENS,
                     overlap=CHUNK_OVERLAP, max_chunks=MAX_RESUME_CHUNKS,
                     source_label='DOCUMENT'):
    sections = split_context_sections(text)
    chunks = []
    for section_label, body in sections:
        for chunk in token_chunk_body(body, tokenizer, max_tokens=max_tokens, overlap=overlap):
            chunks.append(f'[{source_label} SECTION: {section_label}] {chunk}')
    if len(chunks) > max_chunks:
        keep = np.linspace(0, len(chunks) - 1, max_chunks, dtype=int).tolist()
        chunks = [chunks[i] for i in keep]
    fallback = raw_text(text)[:2000] if raw_text(text) else ''
    return chunks or [f'[{source_label} SECTION: GENERAL] {fallback}']

def select_top_chunk_pairs(resume_chunks, jd_chunks, max_pairs=MAX_TRAIN_PAIRS_PER_ROW):
    if len(resume_chunks) * len(jd_chunks) <= max_pairs:
        return [(r, j) for r in resume_chunks for j in jd_chunks]
    try:
        vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=5000)
        mat = vec.fit_transform(resume_chunks + jd_chunks)
        sims = cosine_similarity(mat[:len(resume_chunks)], mat[len(resume_chunks):])
        ranked = np.argsort(sims.reshape(-1))[::-1]
    except Exception:
        ranked = range(len(resume_chunks) * len(jd_chunks))
    selected, seen = [], set()
    for flat_idx in ranked:
        i = int(flat_idx) // len(jd_chunks)
        j = int(flat_idx) % len(jd_chunks)
        if (i, j) in seen:
            continue
        selected.append((resume_chunks[i], jd_chunks[j]))
        seen.add((i, j))
        if len(selected) >= max_pairs:
            break
    return selected or [(resume_chunks[0], jd_chunks[0])]

def build_chunked_examples(frame, max_pairs_per_row=MAX_TRAIN_PAIRS_PER_ROW):
    examples, stats = [], []
    tokenizer = sbert_model.tokenizer
    for _, row in frame.iterrows():
        r_chunks = make_text_chunks(row['resume_raw'], tokenizer, max_chunks=MAX_RESUME_CHUNKS,
                                    source_label='RESUME')
        j_chunks = make_text_chunks(row['jd_raw'], tokenizer, max_chunks=MAX_JD_CHUNKS,
                                    source_label='JOB')
        pairs = select_top_chunk_pairs(r_chunks, j_chunks, max_pairs=max_pairs_per_row)
        label = float(row['ground_truth'])
        examples.extend(InputExample(texts=[r, j], label=label) for r, j in pairs)
        stats.append((len(r_chunks), len(j_chunks), len(pairs)))
    return examples, np.array(stats, dtype=int)

def print_chunk_stats(name, stats):
    if len(stats) == 0:
        return
    print(f'{name}: avg resume chunks={stats[:,0].mean():.2f}, '
          f'avg JD chunks={stats[:,1].mean():.2f}, chunk-pairs={stats[:,2].sum()}')

def aggregate_chunk_similarities(sim_matrix, top_k=TOP_K_CHUNK_PAIRS):
    sims = np.asarray(sim_matrix, dtype=float)
    if sims.size == 0:
        return 0.0
    flat = sims.reshape(-1)
    k = min(top_k, len(flat))
    top_mean = float(np.partition(flat, -k)[-k:].mean())
    coverage = float((sims.max(axis=0).mean() + sims.max(axis=1).mean()) / 2)
    return float(np.clip(0.75 * top_mean + 0.25 * coverage, 0.0, 1.0))

def score_document_pair(model, t1, t2, left_label='RESUME', right_label='JOB'):
    tokenizer = model.tokenizer
    c1 = make_text_chunks(t1, tokenizer, max_chunks=MAX_RESUME_CHUNKS,
                          source_label=left_label)
    c2 = make_text_chunks(t2, tokenizer, max_chunks=MAX_JD_CHUNKS,
                          source_label=right_label)
    e1 = model.encode(c1, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH_SIZE, show_progress_bar=False)
    e2 = model.encode(c2, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH_SIZE, show_progress_bar=False)
    sims = util.cos_sim(e1, e2).detach().cpu().numpy()
    return aggregate_chunk_similarities(sims)

class DocumentSimilarityEvaluator:
    def __init__(self, frame, name='val_doc', save_path=None):
        self.frame = frame.reset_index(drop=True)
        self.name = name
        self.primary_metric = 'pearson'
        self.save_path = save_path
        self.best_score = -np.inf

    def __iter__(self):
        yield self

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        preds = [score_document_pair(model, row['resume_raw'], row['jd_raw'])
                 for _, row in self.frame.iterrows()]
        yt = self.frame['ground_truth'].to_numpy(dtype=float)
        yp = np.asarray(preds, dtype=float)
        if len(yt) < 2 or np.std(yt) == 0 or np.std(yp) == 0:
            pearson, spearman = 0.0, 0.0
        else:
            pearson = pearsonr(yp, yt)[0]
            spearman = spearmanr(yp, yt).correlation
        pearson = 0.0 if np.isnan(pearson) else float(pearson)
        spearman = 0.0 if np.isnan(spearman) else float(spearman)
        mae_norm = mean_absolute_error(yt, yp)
        raw_true = self.frame['ats_score_raw'].to_numpy(dtype=float)
        raw_pred = denormalize_score(yp)
        rmse_raw = float(np.sqrt(np.mean((raw_true - raw_pred) ** 2)))
        print(f'{self.name}: Pearson={pearson:+.4f} Spearman={spearman:+.4f} '
              f'MAE_norm={mae_norm:.4f} RMSE_raw={rmse_raw:.2f}')
        if self.save_path and pearson > self.best_score:
            self.best_score = pearson
            model.save(self.save_path)
            print(f'Saved best document-level checkpoint to {self.save_path}')
        return pearson

train_examples, train_stats = build_chunked_examples(df_tr, MAX_TRAIN_PAIRS_PER_ROW)
print_chunk_stats('Train contextual chunking', train_stats)

val_sample = df_vl.sample(n=min(300, len(df_vl)), random_state=RANDOM_SEED)
val_examples, val_stats = build_chunked_examples(val_sample, MAX_EVAL_PAIRS_PER_ROW)
print_chunk_stats('Validation contextual chunking', val_stats)

train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss   = losses.CosineSimilarityLoss(sbert_model)

doc_val_sample = df_vl.sample(n=min(DOC_EVAL_N, len(df_vl)), random_state=RANDOM_SEED)
evaluator = DocumentSimilarityEvaluator(doc_val_sample, name='val_document', save_path=OUTPUT_PATH)

total_steps = len(train_loader) * EPOCHS
warmup_steps = min(WARMUP_STEPS, max(1, total_steps // 2))
evaluation_steps = max(100, len(train_loader) // 2)

print('Hyperparameters:')
print(f'  epochs={EPOCHS}, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}, warmup_steps={warmup_steps}')
print(f'  chunk_tokens={CHUNK_TOKENS}, overlap={CHUNK_OVERLAP}, train_pairs_per_row={MAX_TRAIN_PAIRS_PER_ROW}')
print(f'Training {len(train_examples)} chunk-pairs for {total_steps} optimizer steps')

sbert_model.fit(
    train_objectives = [(train_loader, train_loss)],
    evaluator        = evaluator,
    epochs           = EPOCHS,
    warmup_steps     = warmup_steps,
    optimizer_params = {'lr': LEARNING_RATE},
    weight_decay     = WEIGHT_DECAY,
    evaluation_steps = evaluation_steps,
    output_path      = None,
    save_best_model  = False,
    show_progress_bar= True,
    use_amp          = (DEVICE == 'cuda')
)

# Reload the best document-level checkpoint. If no evaluation checkpoint was saved,
# save the final model so the reload path is always a valid SentenceTransformer.
if not os.path.exists(os.path.join(OUTPUT_PATH, 'modules.json')):
    print('No valid best checkpoint found; saving final model instead.')
    sbert_model.save(OUTPUT_PATH)
sbert_model = SentenceTransformer(OUTPUT_PATH, device=DEVICE)
sbert_model.max_seq_length = MAX_SEQ_LENGTH
print('Fine-tuning complete. Best contextual chunk-aware model loaded from', OUTPUT_PATH)

In [ ]:
def aggregate_chunk_similarities(sim_matrix, top_k=TOP_K_CHUNK_PAIRS):
    sims = np.asarray(sim_matrix, dtype=float)
    if sims.size == 0:
        return 0.0
    flat = sims.reshape(-1)
    k = min(top_k, len(flat))
    top_mean = float(np.partition(flat, -k)[-k:].mean())
    coverage = float((sims.max(axis=0).mean() + sims.max(axis=1).mean()) / 2)
    return float(np.clip(0.75 * top_mean + 0.25 * coverage, 0.0, 1.0))

def sbert_score(t1, t2, left_label='RESUME', right_label='JOB'):
    return score_document_pair(sbert_model, t1, t2, left_label, right_label)

def tfidf_score(t1, t2):
    try:
        m = TfidfVectorizer().fit_transform([t1, t2])
        return float(cosine_similarity(m[0:1], m[1:2])[0][0])
    except Exception:
        return 0.0

def ensemble_score(s, t, sw=0.85):
    return round((sw*s + (1-sw)*t)*100, 2)

def get_tier(score):
    if score >= 70:   return 'Strong Match'
    elif score >= 45: return 'Potential Fit'
    else:             return 'Poor Match'

def extract_pdf_text(path):
    try:
        return ' '.join(p.get_text() for p in fitz.open(path)).strip()
    except Exception as e:
        return f'PDF error: {e}'

def extract_keywords(text, n=30):
    try:
        vec = TfidfVectorizer(stop_words='english', max_features=n)
        vec.fit([text])
        return set(vec.get_feature_names_out())
    except Exception:
        tokens = word_tokenize(text.lower())
        return set(w for w,_ in Counter(
            t for t in tokens if t not in STOP_WORDS and len(t)>2
        ).most_common(n))

print('Chunk-aware scoring functions ready')

In [ ]:
# Score the full validation set to get meaningful document-level metrics.
EVAL_N    = len(df_vl)
df_sample = df_vl.reset_index(drop=True).copy()
print(f'Scoring all {EVAL_N} validation pairs with fine-tuned model...')

sbert_list, tfidf_list, ens_list = [], [], []
for i, row in df_sample.iterrows():
    s = sbert_score(row['resume_raw'], row['jd_raw'])
    t = tfidf_score(row['resume_clean'], row['jd_clean'])
    sbert_list.append(round(s*100, 2))
    tfidf_list.append(round(t*100, 2))
    ens_list.append(ensemble_score(s, t))
    if (i+1) % 50 == 0 or (i+1) == EVAL_N:
        print(f'  {i+1}/{EVAL_N}')

df_sample['sbert_score']    = sbert_list
df_sample['tfidf_score']    = tfidf_list
df_sample['ensemble_score'] = ens_list
df_sample['sbert_ats_pred'] = denormalize_score(np.asarray(sbert_list) / 100).round(2)
df_sample['tfidf_ats_pred'] = denormalize_score(np.asarray(tfidf_list) / 100).round(2)
df_sample['ensemble_ats_pred'] = denormalize_score(np.asarray(ens_list) / 100).round(2)
df_sample['tier']           = df_sample['ensemble_score'].apply(get_tier)
print('Done.')
df_sample[['sbert_score','tfidf_score','ensemble_score',
           'sbert_ats_pred','tfidf_ats_pred','ensemble_ats_pred','tier']].describe()

In [ ]:
def precision_at_k(yt, yp, k=5, thresh=0.5):
    top = sorted(zip(yp, yt), reverse=True)[:k]
    return sum(1 for _,t in top if t>=thresh) / k

def mrr_score(yt, yp, thresh=0.5):
    for rank, (_,t) in enumerate(sorted(zip(yp,yt), reverse=True), 1):
        if t>=thresh: return 1.0/rank
    return 0.0

def safe_corr(x, y, kind='pearson'):
    if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
        return 0.0
    value = pearsonr(x, y)[0] if kind == 'pearson' else spearmanr(x, y).correlation
    return 0.0 if np.isnan(value) else float(value)

def compute_metrics(yt_norm, yp_percent, name, yt_raw=None):
    yt_norm = np.asarray(yt_norm, dtype=float)
    yp_percent = np.asarray(yp_percent, dtype=float)
    yt_raw = denormalize_score(yt_norm) if yt_raw is None else np.asarray(yt_raw, dtype=float)
    mask = ~np.isnan(yt_norm) & ~np.isnan(yp_percent) & ~np.isnan(yt_raw)
    yt_norm, yp_percent, yt_raw = yt_norm[mask], yp_percent[mask], yt_raw[mask]
    if len(yt_norm) < 2:
        return {}

    yp_norm = np.clip(yp_percent / 100, 0, 1)
    yp_raw = denormalize_score(yp_norm)

    sp = safe_corr(yp_norm, yt_norm, 'spearman')
    pe = safe_corr(yp_norm, yt_norm, 'pearson')
    mae_norm = mean_absolute_error(yt_norm, yp_norm)
    mae_raw = mean_absolute_error(yt_raw, yp_raw)
    rmse_raw = float(np.sqrt(np.mean((yt_raw - yp_raw) ** 2)))
    r2_raw = float(r2_score(yt_raw, yp_raw))
    nd = ndcg_score([yt_norm], [yp_percent])
    p5 = precision_at_k(yt_norm, yp_norm, 5)
    p10 = precision_at_k(yt_norm, yp_norm, 10)
    m = mrr_score(yt_norm, yp_norm)

    print(f'{name:12s}  Spearman={sp:+.4f}  Pearson={pe:+.4f}  '
          f'MAE_norm={mae_norm:.4f}  MAE_raw={mae_raw:.2f}  RMSE_raw={rmse_raw:.2f}  '
          f'R2_raw={r2_raw:+.4f}  NDCG={nd:.4f}  P@5={p5:.2f}  P@10={p10:.2f}  MRR={m:.4f}')
    return dict(model=name, spearman=sp, pearson=pe, mae=mae_norm, mae_norm=mae_norm,
                mae_raw=mae_raw, rmse_raw=rmse_raw, r2_raw=r2_raw, ndcg=nd,
                precision_at_5=p5, precision_at_10=p10, mrr=m)

print('Metrics on full validation set (fine-tuned model):')
yt = df_sample['ground_truth'].values
yt_raw = df_sample['ats_score_raw'].values
r1 = compute_metrics(yt, df_sample['sbert_score'].values,    'SBERT', yt_raw)
r2 = compute_metrics(yt, df_sample['tfidf_score'].values,    'TF-IDF', yt_raw)
r3 = compute_metrics(yt, df_sample['ensemble_score'].values, 'Ensemble', yt_raw)

metrics_df = pd.DataFrame([r1,r2,r3])
if 'model' in metrics_df.columns:
    metrics_df = metrics_df.set_index('model')
print('\nSummary Table:')
display(metrics_df.round(4))

# Targets to compare with the Jina run: RMSE_raw < 8, R2_raw > 0.85,
# MAE_raw < 6, Pearson > 0.9.

In [ ]:
def score_single(resume_text, jd_text, sw):
    s = sbert_score(resume_text, jd_text)
    t = tfidf_score(clean_text(resume_text), clean_text(jd_text))
    return round(s*100,2), round(t*100,2), ensemble_score(s,t,sw)

def tab1_score(resume_file, paste, jd, sw):
    resume = extract_pdf_text(resume_file.name) if resume_file else paste.strip()
    if not resume:     return 'Please upload PDF or paste resume text.', None, ''
    if not jd.strip(): return 'Please enter a job description.', None, ''
    s, t, e = score_single(resume, jd, sw)
    tier    = get_tier(e)
    fig = go.Figure()
    for val,nm,col in zip([s,t,e],['SBERT','TF-IDF','Ensemble'],
                           ['#4A90D9','#E67E22','#27AE60']):
        fig.add_trace(go.Bar(x=[nm], y=[val], marker_color=col,
                             text=[f'{val:.1f}%'], textposition='outside', name=nm))
    fig.update_layout(title='Score Breakdown', yaxis=dict(range=[0,115]),
                      height=350, showlegend=False,
                      plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    kr = extract_keywords(resume, 40)
    kj = extract_keywords(jd,     40)
    matched = kr & kj
    missing = kj - kr
    extra   = kr - kj
    html = (
        "<div style='font-family:sans-serif;padding:12px'>"
        "<h3 style='color:#27AE60'>Matched (" + str(len(matched)) + ")</h3>"
        "<p style='color:#27AE60'>" + (', '.join(sorted(matched)) or 'None') + "</p>"
        "<hr><h3 style='color:#E74C3C'>Missing from Resume (" + str(len(missing)) + ")</h3>"
        "<p style='color:#E74C3C'>" + (', '.join(sorted(missing)) or 'None') + "</p>"
        "<hr><h3 style='color:#F39C12'>Resume-Only Keywords (" + str(len(extra)) + ")</h3>"
        "<p style='color:#F39C12'>" + (', '.join(sorted(extra)) or 'None') + "</p></div>"
    )
    summary = (
        f'## {tier}\n\n'
        f'| Metric | Score |\n|---|---|\n'
        f'| SBERT (fine-tuned) | {s:.1f}% |\n'
        f'| TF-IDF | {t:.1f}% |\n'
        f'| **Ensemble** | **{e:.1f}%** |\n'
        f'| Keyword Overlap | {len(matched)}/{len(kj)} JD keywords |'
    )
    return summary, fig, html

def tab2_rank(files, jd, sw):
    if not files or not jd.strip(): return None, None
    rows = []
    for rf in files:
        text = extract_pdf_text(rf.name)
        name = os.path.basename(rf.name).replace('.pdf','')
        s, t, e = score_single(text, jd, sw)
        rows.append(dict(Candidate=name,SBERT=s,TF_IDF=t,Ensemble=e,Tier=get_tier(e)))
    rdf = pd.DataFrame(rows).sort_values('Ensemble', ascending=False)
    rdf.insert(0,'Rank',range(1, len(rdf)+1))
    colors = ['#27AE60' if r>=70 else '#F39C12' if r>=45 else '#E74C3C'
              for r in rdf['Ensemble']]
    fig = go.Figure(go.Bar(x=rdf['Candidate'], y=rdf['Ensemble'],
                           marker_color=colors,
                           text=[f'{v:.1f}%' for v in rdf['Ensemble']],
                           textposition='outside'))
    fig.update_layout(title='Candidate Ranking', yaxis=dict(range=[0,115]),
                      height=400, plot_bgcolor='rgba(0,0,0,0)',
                      paper_bgcolor='rgba(0,0,0,0)')
    return rdf[['Rank','Candidate','SBERT','TF_IDF','Ensemble','Tier']], fig

def tab3_heatmap(n=20):
    n   = min(int(n), len(df_sample))
    sub = df_sample.head(n).copy()
    sub['Label'] = [f'C{i+1}' for i in range(n)]
    z   = sub[['sbert_score','tfidf_score','ensemble_score']].values.T
    fh  = go.Figure(go.Heatmap(
        z=z, x=sub['Label'].tolist(), y=['SBERT','TF-IDF','Ensemble'],
        colorscale='RdYlGn', zmin=0, zmax=100,
        text=np.round(z,1), texttemplate='%{text}',
        colorbar=dict(title='%')))
    fh.update_layout(title=f'Score Heatmap - {n} Candidates', height=320,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    fd = go.Figure()
    for col,nm,c in zip(['sbert_score','tfidf_score','ensemble_score'],
                         ['SBERT','TF-IDF','Ensemble'],
                         ['#4A90D9','#E67E22','#27AE60']):
        fd.add_trace(go.Histogram(x=df_sample[col],name=nm,
                                   opacity=0.7,marker_color=c,nbinsx=20))
    fd.update_layout(barmode='overlay', title='Score Distribution', height=320,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    return fh, fd

def tab4_compare():
    if metrics_df.empty:
        blank = go.Figure().update_layout(title='No metrics available.')
        return blank, blank, blank
    models  = metrics_df.index.tolist()
    palette = ['#4A90D9','#E67E22','#27AE60']
    mcols   = ['spearman','pearson','ndcg','precision_at_5','precision_at_10','mrr']
    fb = go.Figure()
    for i,m in enumerate(models):
        vals = [metrics_df.loc[m,c] for c in mcols]
        fb.add_trace(go.Bar(name=m,
            x=[c.replace('_',' ').title() for c in mcols],
            y=vals, marker_color=palette[i],
            text=[f'{v:.3f}' for v in vals], textposition='outside'))
    fb.update_layout(barmode='group', title='All Metrics Comparison',
                     yaxis=dict(range=[-0.2,1.3]), height=420,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    rcols = ['spearman','ndcg','precision_at_5','precision_at_10','mrr','pearson']
    fr = go.Figure()
    for i,m in enumerate(models):
        vals = [metrics_df.loc[m,c] for c in rcols]+[metrics_df.loc[m,rcols[0]]]
        cats = [c.replace('_',' ').title() for c in rcols+[rcols[0]]]
        fr.add_trace(go.Scatterpolar(r=vals, theta=cats, fill='toself',
                                      name=m, line_color=palette[i], opacity=0.6))
    fr.update_layout(polar=dict(radialaxis=dict(range=[0,1])),
                     title='Radar Chart', height=420,
                     paper_bgcolor='rgba(0,0,0,0)')
    fm = go.Figure(go.Bar(
        x=models, y=[metrics_df.loc[m,'mae'] for m in models],
        marker_color=palette,
        text=[f"{metrics_df.loc[m,'mae']:.4f}" for m in models],
        textposition='outside'))
    fm.update_layout(title='MAE (lower is better)', height=350,
                     plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    return fb, fr, fm

print('Dashboard functions ready')

In [ ]:
fig_bar_cmp, fig_radar_cmp, fig_mae_cmp = tab4_compare()
fig_heatmap, fig_dist = tab3_heatmap(20)

with gr.Blocks(theme=gr.themes.Soft(), title='EZhire') as demo:

    gr.Markdown('# EZhire - Resume-Job Semantic Similarity Scoring')
    gr.Markdown('Contextual chunk-aware fine-tuned SBERT + TF-IDF ensemble pipeline')

    sbert_w = gr.Slider(0.0, 1.0, value=0.85, step=0.05,
                        label='SBERT Weight  (remainder = TF-IDF weight)')

    with gr.Tab('Score a Resume'):
        with gr.Row():
            with gr.Column():
                r_pdf   = gr.File(label='Upload PDF Resume', file_types=['.pdf'])
                r_paste = gr.Textbox(label='Or paste resume text', lines=7)
                jd_box  = gr.Textbox(label='Job Description', lines=7)
                btn1    = gr.Button('Analyse Match', variant='primary')
            with gr.Column():
                out_md  = gr.Markdown()
                out_fig = gr.Plot(label='Score Breakdown')
        out_kw = gr.HTML(label='Keyword Overlap')
        btn1.click(tab1_score, [r_pdf, r_paste, jd_box, sbert_w],
                   [out_md, out_fig, out_kw])

    with gr.Tab('Rank Candidates'):
        with gr.Row():
            with gr.Column():
                r_multi  = gr.File(label='Upload Multiple PDFs',
                                   file_count='multiple', file_types=['.pdf'])
                jd_rank  = gr.Textbox(label='Job Description', lines=7)
                btn2     = gr.Button('Rank Candidates', variant='primary')
            with gr.Column():
                rank_fig = gr.Plot()
        rank_tbl = gr.Dataframe(label='Ranked Candidates', interactive=False)
        btn2.click(tab2_rank, [r_multi, jd_rank, sbert_w], [rank_tbl, rank_fig])

    with gr.Tab('Score Heatmap'):
        n_sl   = gr.Slider(5, min(50,len(df_sample)), value=20, step=5,
                           label='Candidates to show')
        btn3   = gr.Button('Refresh')
        h_plot = gr.Plot(value=fig_heatmap)
        d_plot = gr.Plot(value=fig_dist)
        btn3.click(tab3_heatmap, [n_sl], [h_plot, d_plot])

    with gr.Tab('Model Comparison'):
        gr.Markdown(
            f'Evaluated on {len(df_sample)} validation samples. '
            f'SBERT was contextual chunk fine-tuned with CosineSimilarityLoss on {len(train_examples):,} chunk-pairs.'
        )
        if not metrics_df.empty:
            gr.Dataframe(value=metrics_df.round(4).reset_index(),
                         label='Metrics Table')
        gr.Plot(value=fig_bar_cmp,   label='All Metrics')
        gr.Plot(value=fig_radar_cmp, label='Radar Chart')
        gr.Plot(value=fig_mae_cmp,   label='MAE')
        gr.Markdown(
            '**Spearman / Pearson**: correlation with ground truth ATS scores (higher = better). '
            '**NDCG**: ranking quality (higher = better). '
            '**Precision@K**: good fits in top K results (higher = better). '
            '**MAE_norm**: normalized prediction error. '
            '**MAE_raw / RMSE_raw / R2_raw**: ATS-scale metrics using the fixed 18.3-90.7 score range.'
        )

demo.launch(share=True, debug=True)